---

# Chapter 1 — Data Importation, Preparation & Validation

---

## 1.0 Environment Setup

All libraries are imported here. 


In [1]:
# ── Core data libraries ────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Statistical testing ────────────────────────────────────────
from scipy import stats
from scipy.stats import (
    f_oneway,       # One-way ANOVA
    chi2_contingency,  # Chi-square test
    ttest_ind,      # Independent samples t-test
    mannwhitneyu,   # Non-parametric alternative
    shapiro         # Normality test
)

# ── Machine learning (Chapter 5) ───────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# ── System ─────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Display settings ───────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 60)

# ── Global plot style ──────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#fafafa',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'grid.linestyle'   : '--',
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
    'axes.labelsize'   : 11,
})

PALETTE     = ['#2E86AB', '#E84855', '#F9C74F', '#43AA8B', '#9B5DE5', '#F77F00']
DENY_COLOR  = '#E84855'   # Red  → denied claims
APPR_COLOR  = '#2E86AB'   # Blue → approved claims

print("✓ All libraries imported successfully.")
print(f"  pandas     {pd.__version__}")
print(f"  numpy      {np.__version__}")
print(f"  matplotlib {plt.matplotlib.__version__}")
print(f"  seaborn    {sns.__version__}")

✓ All libraries imported successfully.
  pandas     2.3.3
  numpy      2.3.5
  matplotlib 3.10.6
  seaborn    0.13.2


---

## 1.1 Path Configuration

Defines all paths once here. If the project folder is moved, only this cell needs updating.


In [2]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.1 — PATH CONFIGURATION
# ═══════════════════════════════════════════════════════════════

import os

# ── Project root ────────
PROJECT_ROOT = os.path.join(os.path.expanduser('~'), 'Desktop',
                            'insurance-claim-fraud-detection')

# ── Data paths ─────────────────────────────────────────────────
DATA_RAW     = os.path.join(PROJECT_ROOT, 'data', 'raw',     'insurance_data.csv')
DATA_EXPORTS = os.path.join(PROJECT_ROOT, 'data', 'exports')

# ── Output paths ───────────────────────────────────────────────
REPORTS_DIR  = os.path.join(PROJECT_ROOT, 'reports')
ASSETS_DIR   = os.path.join(PROJECT_ROOT, 'assets')

# ── Create output directories ──────────────
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(ASSETS_DIR,  exist_ok=True)

# ── Helper function: save figures to reports/ ──────────────────
def save_fig(filename, dpi=180, tight=True):
    """Save current matplotlib figure to reports/ folder."""
    path = os.path.join(REPORTS_DIR, filename)
    if tight:
        plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"  ✓ Saved → reports/{filename}")

# ── Verify paths ───────────────────────────────────────────────
print("Project paths configured:")
print(f"  Raw data   : {DATA_RAW}")
print(f"  Exports    : {DATA_EXPORTS}")
print(f"  Reports    : {REPORTS_DIR}")
print()
print(f"  Raw data file exists : {os.path.exists(DATA_RAW)}")


Project paths configured:
  Raw data   : C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\data\raw\insurance_data.csv
  Exports    : C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\data\exports
  Reports    : C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\reports

  Raw data file exists : False


---

## 1.2 Load the Raw Data

Load directly from the raw CSV exported from the SQL database. This maintains
the full audit trail from PostgreSQL → Python.


In [8]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.2 — LOAD RAW DATA
# Source: insurance_data.csv (Claims Data from mastmustu/Kaggle)
# This is the same file imported into PostgreSQL in Phase 1
# ═══════════════════════════════════════════════════════════════

# Raw data path
DATA_RAW = r"C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\data\insurance_claims.csv"
df_raw = pd.read_csv(DATA_RAW, low_memory=False)

print(f"{'='*55}")
print(f"  Dataset loaded successfully")
print(f"{'='*55}")
print(f"  Rows    : {df_raw.shape[0]:,}")
print(f"  Columns : {df_raw.shape[1]}")
print(f"{'='*55}")
print()

# Preview
df_raw.head(3)


  Dataset loaded successfully
  Rows    : 10,000
  Columns : 38



,TXN_DATE_TIME,TRANSACTION_ID,CUSTOMER_ID,POLICY_NUMBER,POLICY_EFF_DT,LOSS_DT,REPORT_DT,INSURANCE_TYPE,PREMIUM_AMOUNT,CLAIM_AMOUNT,CUSTOMER_NAME,ADDRESS_LINE1,ADDRESS_LINE2,CITY,STATE,POSTAL_CODE,SSN,MARITAL_STATUS,AGE,TENURE,EMPLOYMENT_STATUS,NO_OF_FAMILY_MEMBERS,RISK_SEGMENTATION,HOUSE_TYPE,SOCIAL_CLASS,ROUTING_NUMBER,ACCT_NUMBER,CUSTOMER_EDUCATION_LEVEL,CLAIM_STATUS,INCIDENT_SEVERITY,AUTHORITY_CONTACTED,ANY_INJURY,POLICE_REPORT_AVAILABLE,INCIDENT_STATE,INCIDENT_CITY,INCIDENT_HOUR_OF_THE_DAY,AGENT_ID,VENDOR_ID
0,2020-06-01 00:00:00,TXN00000001,A00003822,PLC00008468,2015-06-23,2020-05-16,2020-05-21,Health,157.13,9000,Christopher Demarest,7701 West Saint John Road,#2010,Glendale,AZ,85308,087-11-1946,Y,54,89,Y,3,L,Own,LI,109134974,HXJP58258181908465,Bachelor,A,Major Loss,Police,0,1,GA,Savannah,4,AGENT00413,VNDR00556
1,2020-06-01 00:00:00,TXN00000002,A00008149,PLC00009594,2018-04-21,2020-05-13,2020-05-18,Property,141.71,26000,Ricardo Gatlin,8595 West 81st Drive,NaN,Arvada,CO,80005,685-33-3536,N,61,80,Y,4,L,Rent,MI,40125819,JUND46859540983731,Bachelor,A,Total Loss,Ambulance,1,0,AL,Montgomery,0,AGENT00769,VNDR00592
2,2020-06-01 00:00:00,TXN00000003,A00003172,PLC00007969,2019-10-03,2020-05-21,2020-05-26,Property,157.24,13000,Lashawn Engles,637 Britannia Drive,NaN,Vallejo,CA,94591,378-36-0672,N,47,68,Y,6,L,Rent,MI,99513168,WGZZ90128415227650,PhD,A,Total Loss,Police,0,1,CO,Grand Junction,19,AGENT00883,VNDR00031


---

## 1.3 Data Types & Initial Inspection

Before any transformation, we inspect what Python infers about each column.


In [9]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.3 — DTYPE INSPECTION
# ═══════════════════════════════════════════════════════════════

print("Column overview:")
print(f"{'Column':<30} {'Dtype':<12} {'Non-Null':>10} {'Null':>8} {'Null%':>8}")
print("─" * 72)

for col in df_raw.columns:
    n_null   = df_raw[col].isnull().sum()
    n_notnull = df_raw[col].notnull().sum()
    null_pct = n_null / len(df_raw) * 100
    print(f"  {col:<28} {str(df_raw[col].dtype):<12} {n_notnull:>10,} {n_null:>8,} {null_pct:>7.1f}%")
    

Column overview:
Column                         Dtype          Non-Null     Null    Null%
────────────────────────────────────────────────────────────────────────
  TXN_DATE_TIME                object           10,000        0     0.0%
  TRANSACTION_ID               object           10,000        0     0.0%
  CUSTOMER_ID                  object           10,000        0     0.0%
  POLICY_NUMBER                object           10,000        0     0.0%
  POLICY_EFF_DT                object           10,000        0     0.0%
  LOSS_DT                      object           10,000        0     0.0%
  REPORT_DT                    object           10,000        0     0.0%
  INSURANCE_TYPE               object           10,000        0     0.0%
  PREMIUM_AMOUNT               float64          10,000        0     0.0%
  CLAIM_AMOUNT                 int64            10,000        0     0.0%
  CUSTOMER_NAME                object           10,000        0     0.0%
  ADDRESS_LINE1                obj

---

## 1.4 Type Casting & Feature Engineering

Cast date columns from strings to proper datetime objects, cast binary columns
to booleans, and engineer derived features that were computed in SQL but are now
available directly in Python for modelling.


In [11]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.4 — TYPE CASTING & FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════════

df = df_raw.copy()

# ── 1. Cast date columns ───────────────────────────────────────
date_cols = ['POLICY_EFF_DT', 'LOSS_DT', 'REPORT_DT']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df['TXN_DATE_TIME'] = pd.to_datetime(df['TXN_DATE_TIME'])

print("✓ Date columns cast to datetime")

# ── 2. Cast binary columns to int (0/1 already, confirm) ──────
df['ANY_INJURY']             = df['ANY_INJURY'].astype(int)
df['POLICE_REPORT_AVAILABLE']= df['POLICE_REPORT_AVAILABLE'].astype(int)

print("✓ Binary columns confirmed as int")

# ── 3. Create binary fraud target variable ─────────────────────
# CLAIM_STATUS: A = Approved (0), D = Denied (1, our fraud proxy)
df['IS_DENIED'] = (df['CLAIM_STATUS'] == 'D').astype(int)

print(f"✓ IS_DENIED created | Denied: {df['IS_DENIED'].sum():,} ({df['IS_DENIED'].mean()*100:.2f}%)")

# ── 4. Derived date features ───────────────────────────────────
# Report lag: days from incident loss to when it was reported
df['REPORT_LAG_DAYS'] = (df['REPORT_DT'] - df['LOSS_DT']).dt.days

# Processing lag: days from report to transaction processing
df['PROCESS_LAG_DAYS'] = (df['TXN_DATE_TIME'].dt.date.map(pd.Timestamp) - df['REPORT_DT']).dt.days

# Total lag: days from loss to transaction
df['TOTAL_LAG_DAYS'] = (df['TXN_DATE_TIME'].dt.date.map(pd.Timestamp) - df['LOSS_DT']).dt.days

# Policy age at loss: how old was the policy when the claim was filed
df['POLICY_AGE_AT_LOSS_DAYS'] = (df['LOSS_DT'] - df['POLICY_EFF_DT']).dt.days

# Vendor involved flag
df['VENDOR_INVOLVED'] = df['VENDOR_ID'].notnull().astype(int)

print("✓ Lag features engineered: REPORT_LAG_DAYS, PROCESS_LAG_DAYS, TOTAL_LAG_DAYS")
print("✓ POLICY_AGE_AT_LOSS_DAYS engineered")
print("✓ VENDOR_INVOLVED flag created")

# ── 5. Handle nulls in CUSTOMER_EDUCATION_LEVEL ───────────────
df['CUSTOMER_EDUCATION_LEVEL'] = df['CUSTOMER_EDUCATION_LEVEL'].fillna('Unknown')
print("✓ CUSTOMER_EDUCATION_LEVEL: 529 nulls → 'Unknown'")

# ── 6. Map coded values to readable labels (for plots) ────────
df['MARITAL_LABEL']    = df['MARITAL_STATUS'].map({'Y': 'Married', 'N': 'Single'})
df['EMPLOYMENT_LABEL'] = df['EMPLOYMENT_STATUS'].map({'Y': 'Employed', 'N': 'Unemployed'})
df['STATUS_LABEL']     = df['CLAIM_STATUS'].map({'A': 'Approved', 'D': 'Denied'})
df['RISK_LABEL']       = df['RISK_SEGMENTATION'].map({'H': 'High', 'M': 'Medium', 'L': 'Low'})
df['CLASS_LABEL']      = df['SOCIAL_CLASS'].map({'HI': 'High Income', 'MI': 'Mid Income', 'LI': 'Low Income'})

print("✓ Readable label columns added for visualisations")
print()
print(f"Final working dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")


✓ Date columns cast to datetime
✓ Binary columns confirmed as int
✓ IS_DENIED created | Denied: 503 (5.03%)
✓ Lag features engineered: REPORT_LAG_DAYS, PROCESS_LAG_DAYS, TOTAL_LAG_DAYS
✓ POLICY_AGE_AT_LOSS_DAYS engineered
✓ VENDOR_INVOLVED flag created
✓ CUSTOMER_EDUCATION_LEVEL: 529 nulls → 'Unknown'
✓ Readable label columns added for visualisations

Final working dataset: 10,000 rows × 49 columns


---

## 1.5 Data Quality Validation

Replicate the SQL audit in Python to confirm the database load was lossless,
and add additional Python-level checks. This section forms the **Data Quality**
subsection of the research methodology.
|

In [12]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.5 — DATA QUALITY VALIDATION
# ═══════════════════════════════════════════════════════════════

print("=" * 60)
print("  DATA QUALITY REPORT")
print("=" * 60)

# ── Row count check ───────────────────────────────────────────
print(f"\n[1] Row count  : {len(df):,}  (Expected: 10,000)")
assert len(df) == 10000, "Row count mismatch!"

# ── Unique key checks ─────────────────────────────────────────
print(f"[2] Unique TRANSACTION_IDs  : {df['TRANSACTION_ID'].nunique():,} (Expected: 10,000)")
print(f"    Unique CUSTOMER_IDs      : {df['CUSTOMER_ID'].nunique():,} (Expected: 10,000)")
print(f"    Unique POLICY_NUMBERs    : {df['POLICY_NUMBER'].nunique():,} (Expected: 10,000)")

# ── Null summary ──────────────────────────────────────────────
print("\n[3] Null counts (expected from SQL audit):")
null_check = {
    'ADDRESS_LINE2'           : 8505,
    'VENDOR_ID'               : 3245,
    'AUTHORITY_CONTACTED'     : 1945,
    'INCIDENT_CITY'           : 46,
    'CITY'                    : 54,
}
for col, expected in null_check.items():
    actual = df_raw[col].isnull().sum()
    status = "✓" if actual == expected else "✗ MISMATCH"
    print(f"    {col:<30}: {actual:,}  (expected {expected:,}) {status}")

# ── Claim status distribution ─────────────────────────────────
print("\n[4] CLAIM_STATUS distribution:")
cs = df['CLAIM_STATUS'].value_counts()
for val, cnt in cs.items():
    label = 'Approved' if val == 'A' else 'Denied (fraud proxy)'
    print(f"    {val} ({label:<22}): {cnt:,}  ({cnt/len(df)*100:.2f}%)")

# ── Numeric range validation ──────────────────────────────────
print("\n[5] Numeric range checks:")
checks = [
    ('PREMIUM_AMOUNT', 6, 200),
    ('CLAIM_AMOUNT',   100, 100000),
    ('AGE',            25, 64),
    ('TENURE',         6, 119),
    ('INCIDENT_HOUR_OF_THE_DAY', 0, 23),
]
for col, lo, hi in checks:
    mn, mx = df[col].min(), df[col].max()
    ok = "✓" if mn >= lo and mx <= hi else "✗"
    print(f"    {col:<32}: min={mn}  max={mx}  {ok}")

# ── Derived feature validation ────────────────────────────────
print("\n[6] Engineered feature checks:")
print(f"    REPORT_LAG_DAYS   : min={df['REPORT_LAG_DAYS'].min()}  max={df['REPORT_LAG_DAYS'].max()}  mean={df['REPORT_LAG_DAYS'].mean():.2f}")
print(f"    PROCESS_LAG_DAYS  : min={df['PROCESS_LAG_DAYS'].min()}  max={df['PROCESS_LAG_DAYS'].max()}  mean={df['PROCESS_LAG_DAYS'].mean():.2f}")
print(f"    VENDOR_INVOLVED   : {df['VENDOR_INVOLVED'].sum():,} claims involved a vendor ({df['VENDOR_INVOLVED'].mean()*100:.1f}%)")

print("\n" + "=" * 60)
print("  ✓ All quality checks passed. Data ready for analysis.")
print("=" * 60)


  DATA QUALITY REPORT

[1] Row count  : 10,000  (Expected: 10,000)
[2] Unique TRANSACTION_IDs  : 10,000 (Expected: 10,000)
    Unique CUSTOMER_IDs      : 10,000 (Expected: 10,000)
    Unique POLICY_NUMBERs    : 10,000 (Expected: 10,000)

[3] Null counts (expected from SQL audit):
    ADDRESS_LINE2                 : 8,505  (expected 8,505) ✓
    VENDOR_ID                     : 3,245  (expected 3,245) ✓
    AUTHORITY_CONTACTED           : 1,945  (expected 1,945) ✓
    INCIDENT_CITY                 : 46  (expected 46) ✓
    CITY                          : 54  (expected 54) ✓

[4] CLAIM_STATUS distribution:
    A (Approved              ): 9,497  (94.97%)
    D (Denied (fraud proxy)  ): 503  (5.03%)

[5] Numeric range checks:
    PREMIUM_AMOUNT                  : min=6.0  max=200.0  ✓
    CLAIM_AMOUNT                    : min=100  max=100000  ✓
    AGE                             : min=25  max=64  ✓
    TENURE                          : min=6  max=119  ✓
    INCIDENT_HOUR_OF_THE_DAY        

---

## 1.6 Dataset Summary Statistics

A structured statistical summary of all key variables. This table will be referenced
in the research report as **Table 1: Descriptive Statistics**.


In [13]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.6 — DESCRIPTIVE STATISTICS
# ═══════════════════════════════════════════════════════════════

print("TABLE 1: Descriptive Statistics — Continuous Variables")
print("=" * 70)

num_cols = [
    'PREMIUM_AMOUNT', 'CLAIM_AMOUNT', 'AGE', 'TENURE',
    'NO_OF_FAMILY_MEMBERS', 'INCIDENT_HOUR_OF_THE_DAY',
    'REPORT_LAG_DAYS', 'PROCESS_LAG_DAYS', 'POLICY_AGE_AT_LOSS_DAYS'
]

desc = df[num_cols].describe().T
desc.columns = ['N','Mean','Std','Min','Q1','Median','Q3','Max']
desc['N'] = desc['N'].astype(int)
print(desc.round(2).to_string())

print()
print("TABLE 2: Descriptive Statistics — Categorical Variables")
print("=" * 70)

cat_cols = {
    'INSURANCE_TYPE'           : 'Insurance Type',
    'RISK_SEGMENTATION'        : 'Risk Segment (H/M/L)',
    'SOCIAL_CLASS'             : 'Social Class (HI/MI/LI)',
    'HOUSE_TYPE'               : 'House Type',
    'MARITAL_LABEL'            : 'Marital Status',
    'EMPLOYMENT_LABEL'         : 'Employment Status',
    'CUSTOMER_EDUCATION_LEVEL' : 'Education Level',
    'INCIDENT_SEVERITY'        : 'Incident Severity',
    'CLAIM_STATUS'             : 'Claim Status (Target)',
}

for col, label in cat_cols.items():
    counts = df[col].value_counts()
    parts  = [f"{v}: {n:,} ({n/len(df)*100:.1f}%)" for v, n in counts.items()]
    print(f"  {label:<30} → {' | '.join(parts)}")
    

TABLE 1: Descriptive Statistics — Continuous Variables
                              N      Mean       Std    Min       Q1   Median        Q3        Max
PREMIUM_AMOUNT            10000     88.51     48.32   6.00    62.02    88.84    121.90     200.00
CLAIM_AMOUNT              10000 16,563.83 22,037.49 100.00 2,000.00 7,000.00 21,000.00 100,000.00
AGE                       10000     44.47     11.52  25.00    34.00    45.00     54.00      64.00
TENURE                    10000     62.95     32.89   6.00    34.00    64.00     92.00     119.00
NO_OF_FAMILY_MEMBERS      10000      4.49      1.71   2.00     3.00     4.00      6.00       7.00
INCIDENT_HOUR_OF_THE_DAY  10000     11.60      6.88   0.00     6.00    12.00     18.00      23.00
REPORT_LAG_DAYS           10000      3.21      1.97   0.00     1.00     3.00      5.00       5.00
PROCESS_LAG_DAYS          10000      7.21      4.07   1.00     4.00     6.00     11.00      15.00
POLICY_AGE_AT_LOSS_DAYS   10000  2,096.48  1,029.52 110.00 1,21

---

## 1.7 Chapter 1 Summary

| Step | Action | Outcome |
|---|---|---|
| 1.2 | Loaded raw CSV | 10,000 rows × 38 columns confirmed |
| 1.3 | Inspected dtypes | Date columns read as strings — requires casting |
| 1.4 | Type casting | 4 date columns → datetime; binary confirmed |
| 1.4 | Target variable | `IS_DENIED` created (1 = Denied, 0 = Approved) |
| 1.4 | Lag features | `REPORT_LAG_DAYS`, `PROCESS_LAG_DAYS`, `TOTAL_LAG_DAYS` |
| 1.4 | Policy age | `POLICY_AGE_AT_LOSS_DAYS` derived |
| 1.4 | Vendor flag | `VENDOR_INVOLVED` (1 if vendor assigned) |
| 1.5 | Quality validation | All SQL audit results confirmed in Python |
| 1.6 | Descriptive stats | Tables 1 & 2 generated for research report |

**Next:** Chapter 2 — Exploratory Data Analysis (EDA)

---
